In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.integrate import nquad

In [2]:
DATA_PATH = Path("../DATA/Ar_Stopping_Power_NIST_ASTAR.csv")

In [3]:
def load_astar_table(Path):
    df = pd.read_csv(Path, header=None, skiprows=8)
    df = df.dropna()
    energies = df[0].values
    csda = df[1].values
    return energies, csda

In [4]:
ENERGIES, CSDA = load_astar_table(DATA_PATH)

In [5]:
def range_csda(E_mev: float) -> float:
    """
    Interpolate the CSDA range [g/cm^2] at a given energy E_mev [MeV].
    """
    R_csda = np.interp(E_mev, ENERGIES, CSDA)
    return float(R_csda)


def argon_density_gcm3(pressure_bar: float) -> float:
    """
    Argon density [g/cm^3] at room temperature, scaled linearly with pressure.
    rho_STP ≈ 1.784e-3 g/cm^3 at 1 bar.
    """
    rho_stp = 1.784e-3  # g/cm^3 at ~1 bar, room temp
    rho = rho_stp * pressure_bar  # ideal gas scaling
    return rho

def range_cm(E_mev: float, pressure_bar: float) -> float:
    """
    Mean track length [cm] of an alpha with energy E_mev [MeV] in argon gas
    at given pressure_bar, using NIST CSDA and ideal-gas density scaling.
    """
    R_csda = range_csda(E_mev)              # g/cm^2
    rho = argon_density_gcm3(pressure_bar)  # g/cm^3
    return float(R_csda / rho)              # cm

In [6]:
E_alpha = 1   # MeV
P = 1          # bar

L_cm = range_cm(E_alpha, P)
print(f"Mean track length at {P} bar: {L_cm:.3f} cm")

Mean track length at 1 bar: 0.617 cm


In [7]:
def Photons_Per_Electron(EL, P):
    return [81 * EL + 47] * P * .7

def Electrons_Per_Energy_Track(E):
    return E/26

In [8]:
def omega_point(y0, R_w):

    return 2.0 * np.pi * (1.0 - y0 / np.sqrt(y0**2 + R_w**2))

In [9]:
def omega_avg_nquad(y0, R_EL, R_w):
    def integrand(z_w, x_w, z_s, x_s):
        denom = ((x_w - x_s)**2 + (z_w - z_s)**2 + y0**2)**1.5
        return 1.0 / denom 

    def limits_z_w(x_w, z_s, x_s):
        return [-np.sqrt(R_w**2 - x_w**2), np.sqrt(R_w**2 - x_w**2)]

    def limits_x_w(z_s, x_s):
        return [-R_w, R_w]

    def limits_z_s(x_s):
        return [-np.sqrt(R_EL**2 - x_s**2), np.sqrt(R_EL**2 - x_s**2)]

    def limits_x_s():
        return [-R_EL, R_EL]

    result, err = nquad(
        integrand,
        [limits_z_w, limits_x_w, limits_z_s, limits_x_s]
    )

    omega_avg = (y0 / (np.pi * R_EL**2)) * result
    return omega_avg


In [10]:
y0   = 56.47*25.4-190-125-390
R_EL = 419/2 
R_w  = 25.4
#y0 = 3*25.4
#R_EL = 5
#R_w = 25.4

print (y0)

729.338


In [11]:
Omega_point = omega_point(y0, R_w)
Omega_avg   = omega_avg_nquad(y0, 10*L_cm, R_w)

print("Ω_point =", Omega_point)
print("<Ω>     =", Omega_avg)
print("fraction (avg) =", Omega_avg / (4.0 * np.pi))

Ω_point = 0.0038068414452069642
<Ω>     = 0.0038066456737869407
fraction (avg) = 0.00030292323779128506


In [12]:
489*(1e6*E_alpha/26)*Omega_avg /(4*np.pi)/2/4

712.16088115355

In [13]:
print(f"Collection efficiency (%) = {(Omega_avg / (4*np.pi)):.7f}")

Collection efficiency (%) = 0.0003029


In [14]:
print(f"Collection efficiency (%) = {100 * (489*(1e6*E_alpha/26)*Omega_avg /(4*np.pi)) / (489*(1e6*E_alpha/26)):.3f}")

Collection efficiency (%) = 0.030


In [15]:
f = 100
u = 1438.148-190-80
v = f*u/(u-f)
print(u)
print(v)

1168.148
109.36199852454904


In [16]:
d = v-50
theta = np.arctan(3.250/5.640)
x = d * np.sin(theta)
y = d * np.cos(theta)
print(x/10,y/10)

2.9638203831367544 5.143368295658859


In [17]:
import numpy as np

# -----------------------------
# Am-241 alpha lines (MeV, branching)
# -----------------------------

AM241_ALPHA_LINES = [
    (5.48556, 0.848),
    (5.44280, 0.131),
    (5.38800, 0.0166),
]

# -----------------------------
# Physics input
# -----------------------------

# Effective energy per photon (eV/photon)
# For 1 bar gas, not well-defined → treat as tunable
W_SC = 60.0  # try 50–70 eV range

# Fraction of alpha energy deposited in gas
# 1.0 = fully stops in gas
# <1.0 if it escapes or originates in substrate
ALPHA_DEPOSITION_FRACTION = 1.0

# -----------------------------
# Core calculation
# -----------------------------

def mean_alpha_energy(alpha_lines):
    return sum(E * br for E, br in alpha_lines)

def photons_per_decay(alpha_lines, W_sc, deposition_fraction=1.0):
    E_mean = mean_alpha_energy(alpha_lines)
    E_dep = E_mean * deposition_fraction  # MeV

    N_gamma = (E_dep * 1e6) / W_sc  # convert MeV → eV

    return {
        "mean_alpha_energy_MeV": E_mean,
        "deposited_energy_MeV": E_dep,
        "photons_generated": N_gamma
    }

# -----------------------------
# Run
# -----------------------------

result = photons_per_decay(
    AM241_ALPHA_LINES,
    W_sc=W_SC,
    deposition_fraction=ALPHA_DEPOSITION_FRACTION
)

print(f"Mean alpha energy: {result['mean_alpha_energy_MeV']:.4f} MeV")
print(f"Deposited energy:  {result['deposited_energy_MeV']:.4f} MeV")
print(f"Photons generated: {result['photons_generated']:.0f}")

Mean alpha energy: 5.4542 MeV
Deposited energy:  5.4542 MeV
Photons generated: 90903
